# Evaluation Metrics for Poses and Trajectories

ATE, RPE, and the challenge-style **AA / mAA** are not three unrelated metrics.
Two of them are the *same residual* aggregated differently; the third is
deliberately different. This notebook is the bridge between the VO/SLAM
evaluation vocabulary and the Structure-from-Motion benchmark vocabulary.

| Metric | Tool | Meaning |
| ------- | --------- | ---------------------------------------- |
| **ATE** (APE) | `evo_ape` | Absolute Trajectory Error — global drift |
| **RPE** | `evo_rpe` | Relative Pose Error — local accuracy |
| **AA / mAA** | IMC scorer | Fraction of cameras inside a distance threshold, averaged over thresholds |

## 1. The one residual behind ATE and mAA

Both a monocular VO trajectory and an SfM reconstruction are only defined **up to
a similarity transform** — arbitrary origin, orientation, and scale. So neither
can be compared to ground truth directly. Both first solve

$$
T^\star = \arg\min_{s,R,\mathbf{t}} \sum_i \big\lVert\, C_{g,i} - (s R\,C_i + \mathbf{t}) \,\big\rVert^2 ,
\qquad s>0,\; R \in SO(3),
$$

then measure the **same per-camera residual**

$$
r_i \;=\; \big\lVert\, C_{g,i} - T^\star(C_i) \,\big\rVert .
$$

Everything after that is just a choice of how to summarise the $r_i$:

| | Summary of $\{r_i\}$ | Units | Hides |
|---|---|---|---|
| **ATE** | RMSE / median / max | metres | the shape of the distribution |
| **AA at threshold $\tau$** | $\dfrac{\#\{i : r_i < \tau\}}{N}$ | fraction | how badly the failures failed |
| **mAA** | mean of $A(\tau)$ over $\tau \in \mathcal{T}$ | fraction | same |

So, in one sentence:

> **mAA is the empirical CDF of the ATE residual, sampled at a few thresholds and averaged.**

ATE gives one number in metres and is dominated by the worst cameras. AA is
bounded in $[0,1]$, insensitive to how catastrophic an outlier is, and therefore
much better behaved as a leaderboard score — a single wildly misregistered camera
costs you $1/N$, not an unbounded RMSE.

**RPE is the odd one out.** It never applies a global alignment; it compares
relative motions over a window $\Delta$, which makes it gauge-invariant by
construction. There is no RPE analogue in SfM benchmarking, because an
unordered image collection has no temporal ordering to define $\Delta$.

### 1.1 Worked example — the same residuals, two scores

Ten cameras. After Sim(3) alignment, the distance between each estimated and true camera centre is:

$$
\{0.12,\;0.05,\;0.31,\;0.44,\;0.09,\;0.62,\;1.40,\;0.27,\;0.18,\;6.00\}\ \text{m}
$$

Nine are good; one is a catastrophic 6 m outlier.

**As ATE** — square, average, square-root:

$$
\mathrm{RMSE}=\sqrt{\tfrac{0.0144+0.0025+0.0961+0.1936+0.0081+0.3844+1.96+0.0729+0.0324+36}{10}}=\sqrt{3.877}=\mathbf{1.97\ \text{m}}
$$

The single 6 m camera contributes 36 of the 38.8 total — **93 % of the score comes from one camera out of ten**.

**As mAA** — count how many fall under each threshold:

| $\tau$ | cameras with $e_i<\tau$ | $\mathrm{AA}(\tau)$ |
|---|---|---|
| 0.5 m | 7 | 70 % |
| 1 m | 8 | 80 % |
| 2 m | 9 | 90 % |
| 4 m | 9 | 90 % |

$$
\mathrm{mAA}=\tfrac{70+80+90+90}{4}=\mathbf{82.5\,\%}
$$

Identical residuals, opposite verdicts: ATE says "2 m accuracy" (bad), mAA says "82 %" (respectable). Thresholding caps what any one camera can cost you at $1/N$; squaring lets it dominate.

## 2. Same Umeyama, different robustness

The closed form for $T^\star$ — centroids, cross-covariance, SVD, scale,
translation — is identical in both worlds (Umeyama 1991 / Horn 1987). It is
derived in
[Trajectory analysis §4.1](visual_odometry/trajectory_analysis.ipynb#41-sim3--umeyama-1991-for-monocular).

What differs is **how it is fitted**:

| | Input | Fit |
|---|---|---|
| `evo` / `rpg_trajectory_evaluation` | ordered, timestamped trajectory | plain least squares over all poses |
| IMC scorer | unordered bag of camera centres | **RANSAC** over triplets, keep the model with most inliers |

This is not cosmetic. A VO trajectory drifts *smoothly*, so every pose is roughly
consistent and least squares is fine. An SfM reconstruction fails
*discontinuously* — mirrored sub-clusters, cameras dumped at the origin,
whole components misregistered — and a non-robust fit gets dragged off by a
handful of them, destroying the score for cameras that were actually correct.

If you ever evaluate an SfM result with `evo`-style least-squares alignment, you
will under-report accuracy for exactly this reason.

### 2.1 Worked example — why least squares loses the good cameras

Five cameras along a line. The reconstruction is **exact** for four of them; the fifth blew up and landed at $x=100$. Only a translation needs fitting:

| camera | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| ground truth $x$ | 0 | 1 | 2 | 3 | 4 |
| estimate $x$ | 0 | 1 | 2 | 3 | **100** |

**Least squares** puts the centroids on top of each other:

$$
t = \bar{x}^{gt}-\bar{x}^{est} = 2 - \frac{0+1+2+3+100}{5} = 2 - 21.2 = -19.2
$$

Every camera is shifted by $-19.2$ m, so the four *perfect* cameras now sit 19.2 m from truth. At any threshold below 19.2 m, **not one camera counts as correct — mAA = 0 %.**

**RANSAC** proposes $t$ from single cameras and keeps the hypothesis with the most inliers. The hypothesis $t=0$ agrees with cameras 1–4 exactly:

| errors with $t=0$ | 0 | 0 | 0 | 0 | 96 |
|---|---|---|---|---|---|

Four inliers, one rejected outlier — **mAA = 80 %** at every threshold.

Same reconstruction, same metric, same alignment *formula*. The only difference is how the transform was fitted, and it moves the score from 0 % to 80 %.

## 3. The convention trap: $C = -R^\top t$

Every one of these metrics is defined on **camera centres in the world frame**,
not on the translation vector you store.

$$
X_{\text{cam}} = R\,X_{\text{world}} + t
\qquad\Longrightarrow\qquad
C = -R^{\top} t .
$$

A submission or a trajectory file that puts $t$ where $C$ belongs is not
"slightly off" — it is wrong by the full camera position, and scores ~0 while
looking perfectly well-formed. The world→cam vs cam→world bookkeeping is worked
through in [Understanding poses](understanding_poses.md).

### 3.1 Worked example — a 4.2 m error on a perfect pose

One camera, 3 m east of the world origin, yawed $90^\circ$. Its world→cam rotation and true centre:

$$
R=\begin{pmatrix}0&1&0\\-1&0&0\\0&0&1\end{pmatrix},
\qquad
\mathbf{C}=(3,\,0,\,0).
$$

What a pose file actually stores is $\mathbf{t}$, not $\mathbf{C}$:

$$
\mathbf{t}=-R\,\mathbf{C}
=-\begin{pmatrix}0&1&0\\-1&0&0\\0&0&1\end{pmatrix}\begin{pmatrix}3\\0\\0\end{pmatrix}
=-\begin{pmatrix}0\\-3\\0\end{pmatrix}=(0,\,3,\,0).
$$

**Read $\mathbf{t}$ as the camera centre** and you place the camera at $(0,3,0)$ instead of $(3,0,0)$:

$$
\|(0,3,0)-(3,0,0)\| = \sqrt{9+9} = \mathbf{4.24\ \text{m}}
$$

— a 4.24 m error on a pose that is *exactly right*. **Recover the centre properly**:

$$
-R^\top\mathbf{t}
=-\begin{pmatrix}0&-1&0\\1&0&0\\0&0&1\end{pmatrix}\begin{pmatrix}0\\3\\0\end{pmatrix}
=-\begin{pmatrix}-3\\0\\0\end{pmatrix}=(3,\,0,\,0)=\mathbf{C}. \quad\checkmark
$$

Note the failure is silent: with a metric threshold of 0.5–4 m, this camera is scored as wrong, and the bug looks exactly like a reconstruction failure. Only the $90^\circ$ yaw exposes it — a camera at the origin, or with identity rotation, gives $-R^\top\mathbf{t}=\mathbf{t}$ and the bug hides completely.

## 4. Two different metrics are both called "mAA"

This is the single most common source of confusion when reading matching papers,
because the Image Matching Challenge changed families mid-life and kept the name.

| | **Relative-pose mAA** (IMC ~2019–2023) | **Camera-centre mAA** (IMC 2024, 2025) |
|---|---|---|
| Unit of evaluation | image **pair** | individual **camera** |
| Error measured | rotation geodesic on $SO(3)$, in degrees, **and** translation-*direction* angle in $[0,180]^\circ$ | Euclidean distance between 3D centres, in metres |
| Thresholds | angular, e.g. $10^\circ$ | metric, e.g. $\{0.5,1,2,4\}$ m |
| Global alignment | none | Sim(3), RANSAC-fitted |
| Scale | invisible (unit vectors) | explicitly recovered as $s$ |
| Combination rule | $\max(\text{err}_{\text{rot}}, \text{err}_{\text{trans}}) < \tau$ | single distance threshold |

Both are "mean Average Accuracy" — cumulative accuracy at increasing thresholds,
averaged. Only the error function changes.

Note that the rotation error of the *first* family,

$$
\theta = \arccos\!\Big(\frac{\operatorname{tr}(R_a^{\top} R_b) - 1}{2}\Big),
$$

is the **same geodesic distance** used as a *training* loss in
[VO loss functions §1.1](vo_loss_functions.ipynb) — measured in degrees for
evaluation, minimised in radians for training.

**Where each is written up:**

- Relative-pose mAA, and its relationship to mAP —
  [PythonTutorial · metrics_and_scoring_methods.ipynb §1.7.1](https://github.com/behnamasadi/PythonTutorial/blob/master/machine_learning/metrics_and_scoring_for_model_evaluation/metrics_and_scoring_methods.ipynb#171-maa-mean-average-accuracy)
- Camera-centre mAA, with clustering and a fully worked Umeyama alignment —
  `PyTorchTutorial/kaggle_structure/competitions/image_matching_2025/imc2025_scoring_explained.ipynb`

### 4.1 Worked example — one reconstruction, two verdicts

A single image pair. The estimated relative pose is off by $8^\circ$ in rotation and $12^\circ$ in translation *direction*; after Sim(3) alignment the two camera centres land $0.8$ m from truth.

**Relative-pose mAA** (IMC ~2019–2023) takes the **worse** of the two angles and compares it to an angular threshold:

$$
\mathrm{err}=\max(8^\circ,\;12^\circ)=12^\circ \;>\; \tau=10^\circ
\quad\Longrightarrow\quad \textbf{fail}
$$

**Camera-centre mAA** (IMC 2024–2025) never looks at angles at all:

$$
0.8\ \text{m} \;<\; \tau=1\ \text{m} \quad\Longrightarrow\quad \textbf{pass}
$$

Same estimate, same challenge name, opposite result. Note also what each is blind to: the relative-pose family works on unit translation vectors, so a reconstruction at **half the true scale** scores perfectly — the direction is unchanged. The camera-centre family recovers $s$ explicitly during alignment, so it would report that same reconstruction as badly wrong at every metric threshold. When a paper quotes "mAA", the number is meaningless until you know which family, which thresholds, and which year.


## 5. What is *not* related

The training-side notebooks in this repo — [photometric reprojection
loss](photometric_reprojection_loss.ipynb), [SSIM](ssim.ipynb), [edge-aware depth
smoothness](edge_aware_depth_smoothness.ipynb) — are dense, differentiable,
**image-space** objectives used to train a depth+pose network.

None of them appear in any of the metrics above. A benchmark scorer receives
poses, never pixels. The only connection is a pipeline one: those losses train a
network whose poses you could then evaluate with ATE or submit to a challenge.
(In practice the challenge leaderboards are dominated by classical SfM —
matching plus COLMAP/GLOMAP — not by learned VO.)

The pose-side losses are the exception: the $SO(3)$ geodesic and SE(3) losses in
[VO loss functions](vo_loss_functions.ipynb) are the same functions as the
evaluation errors above, just used as gradients instead of as reports.

## 6. See also

- [Trajectory analysis](visual_odometry/trajectory_analysis.ipynb) — ATE/RPE in
  depth, Sim(3)/SE(3)/yaw-only/posyaw alignment, sub-trajectory drift, and the
  `evo` / `rpg_trajectory_evaluation` tooling
- [Understanding poses](understanding_poses.md) — KITTI pose format, $T_{w,i}$,
  and the world→cam conversion
- [VO loss functions](vo_loss_functions.ipynb) — the training-time counterparts
- [Estimator consistency](visual_odometry/estimator_consistency.ipynb) — NEES,
  NIS, the gauge trap, and why a covariance nobody audits is the failure mode
  ATE structurally cannot see
- [Benchmark methodology](visual_odometry/benchmark_methodology.ipynb) —
  run-to-run nondeterminism, paired significance tests, and the aggregation
  choices that silently change a ranking
- [Runtime evaluation](visual_odometry/runtime_evaluation.ipynb) — latency vs
  throughput, tail costs, memory and energy
